# Session 6: MLflow Projects & Model Registry
## Part 2: Model Registry - Managing Production Model Lifecycle

**MLOps with Agentic AI - Advanced Certification Course**

---

### Welcome Back from Break!

**Part 1 Recap (What we covered):**
- MLflow Projects for standardizing ML workflows
- Converting Session 5 code into reusable Projects
- Running projects with different parameters
- Entry points and conda environments

**Part 2 (What we'll cover now):**
- MLflow Model Registry for production model management
- Registering your Session 5 models
- Model lifecycle stages (Staging -> Production)
- Version comparison and rollback strategies
- Approval workflows and governance

---

### The Complete MLflow Journey

```
Session 5: Track Experiments
    ↓
Part 1: Standardize with Projects  (DONE)
    ↓
Part 2: Operationalize with Registry  (YOU ARE HERE)
    ↓
Deploy to Production
```

---

## Section 1: Introduction to Model Registry (15 min)

### The Problem: Model Management Chaos

Imagine this scenario...

### Real-World Scenario

**You (Data Scientist):**  
"We trained 15 models in Session 5. Which one is in production?"

**Your Teammate:**  
"Umm... I think it's the one from last Tuesday? Or was it Wednesday?"

**DevOps Engineer:**  
"I'm serving run_id abc123xyz... is that the right one?"

**Manager:**  
"The production model is failing! Which version should we rollback to?"

**Everyone:** Confused and stressed!

---

**The Reality:**
- No single source of truth for production models
- Manual tracking in spreadsheets or Slack messages
- No clear approval workflow
- Difficult to rollback when things go wrong
- No audit trail for compliance
- Can't compare model versions easily

**This chaos happens in EVERY ML team without proper model management!**

### The Solution: MLflow Model Registry

MLflow Model Registry is a **centralized model store** for managing the full lifecycle of ML models.

**Think of it as:**
- **Git for models** - Version control for ML models
- **App Store for models** - Central repository with metadata
- **Traffic control** - Manage what goes to production
- **Audit system** - Track who approved what and when

---

### What Model Registry Provides

1. **Single Source of Truth**
   - One place to find all registered models
   - Clear visibility of what's in production

2. **Automatic Versioning**
   - Every model gets a version number
   - Easy to track model evolution

3. **Lifecycle Stages**
   - None -> Staging -> Production -> Archived
   - Clear workflow for model promotion

4. **Model Lineage**
   - Link back to training run and data
   - Understand model provenance

5. **Metadata & Governance**
   - Descriptions, tags, annotations
   - Approval workflows and audit trails

6. **Easy Deployment**
   - Load models by stage: `models:/MyModel/Production`
   - No more hardcoding run IDs!

### Benefits of Model Registry

| Without Registry | With Registry |
|------------------|---------------|
| "Which model is in production?" | Clear visibility in one place |
| Manual spreadsheet tracking | Automated version control |
| Slack messages for approvals | Built-in approval workflows |
| Chaotic rollbacks | One-command rollback |
| No audit trail | Complete history & compliance |
| Hardcoded model paths | Load by stage dynamically |

---

### Model Lifecycle Stages

Every model in the Registry goes through stages:

```
None (Newly Registered)
      ↓
Staging (Testing & Validation)
      ↓
Production (Serving Live Traffic)
      ↓
Archived (Retired, Kept for Audit)
```

**Stage Definitions:**

- **None**: Just registered, not yet validated
- **Staging**: Being tested, pending approval
- **Production**: Currently serving predictions
- **Archived**: Retired but kept for compliance

**Pro Tip:** Always test in Staging before promoting to Production!

### When to Use Model Registry

**Use Registry When:**
- Deploying models to production
- Multiple people working on same models
- Need approval workflows
- Compliance and audit requirements
- Managing multiple model versions
- Need rollback capabilities

**Don't Need Registry For:**
- Quick experiments in notebooks
- Personal research projects
- One-off analyses

**Pro Tip:** If it touches production, it should be in the Registry!

---

## Section 2: Setting Up Model Registry (10 min)

### Prerequisites Check

Before we start, let's verify your setup:

In [1]:
# Verify imports and setup
import mlflow
from mlflow.tracking import MlflowClient
import pandas as pd
import json
from pathlib import Path

print("All imports successful!")
print(f"\nMLflow version: {mlflow.__version__}")

# # Check if registry database exists
# registry_db = Path("mlflow_registry.db")
# if registry_db.exists():
#     print(f"\nRegistry database found: {registry_db.absolute()}")
# else:
#     print(f"\nRegistry database not found!")
#     print(f"   Please run: python setup/setup_registry.py")

All imports successful!

MLflow version: 2.9.2


### Configure Tracking URI

The Model Registry requires a database backend. We'll use SQLite for local development:

In [2]:
# Set tracking URI to use registry database
# tracking_uri = "sqlite:///mlflow_registry.db"
# mlflow.set_tracking_uri(tracking_uri)

# print(f"Tracking URI set to: {tracking_uri}")
# print(f"\nThis URI supports:")
# print(f"   - Experiment tracking (from Session 5)")
# print(f"   - Model Registry (new in Session 6!)")

mlflow.set_tracking_uri("file:./mlruns")

# Verify connection
client = MlflowClient()
print(f"\nConnected to MLflow Registry!")


Connected to MLflow Registry!


### Load Your Session 5 Best Models

Remember `session5_best_runs.json`? Let's load it!

In [3]:
# Load best runs from Session 5
with open('session5_best_runs.json', 'r') as f:
    best_runs = json.load(f)

print("Your Top 3 Models from Session 5:\n")

best_runs = best_runs['top_runs']

# print(f"Best Runs: {best_runs['top_runs']}")

for run in best_runs:
    print(f"Rank {run['rank']}:")
    print(f"  - Run ID: {run['run_id'][:16]}...")
    print(f"  - Accuracy: {run['accuracy']:.4f}")
    print(f"  - Model: {run.get('model_type', 'N/A')}")
    # print(f"  - Experiment: {run['experiment_name']}")
    print()

print("We'll register these models in the Registry!")

# Store best run ID for easy access
best_run_id = best_runs[0]['run_id']
print(f"\nYour best model run_id: {best_run_id}")

Your Top 3 Models from Session 5:

Rank 1:
  - Run ID: 8cc92639f5374bd0...
  - Accuracy: 0.6835
  - Model: N/A

Rank 2:
  - Run ID: a0daa66b52184b36...
  - Accuracy: 0.6830
  - Model: N/A

Rank 3:
  - Run ID: 796bad712edf4a9b...
  - Accuracy: 0.6820
  - Model: N/A

We'll register these models in the Registry!

Your best model run_id: 8cc92639f5374bd0a1710e84c381b86a


In [4]:
best_run = best_runs[0]

print("🎯 Best Model Details:\n")
print(f"Rank: {best_run['rank']}")
print(f"Run ID: {best_run['run_id']}")
print(f"Accuracy: {best_run['accuracy']:.4f}")

print("\n💡 We'll register this as our first model!")

🎯 Best Model Details:

Rank: 1
Run ID: 8cc92639f5374bd0a1710e84c381b86a
Accuracy: 0.6835

💡 We'll register this as our first model!


---

## Section 3: 🚨 DETECT & FIX Parent-Child Issue

🔍 CRITICAL CHECK: Do our runs have model artifacts?

Before registering, let's verify the runs actually have models!

In [5]:
def check_model_artifact(run_id, artifact_path="best_model"):
    """Check if a run has a model artifact"""
    try:
        artifacts = client.list_artifacts(run_id, artifact_path)
        return len(artifacts) > 0
    except Exception:
        return False

print("🔍 Checking if our best runs have model artifacts...\n")

issues_found = []

for i, run in enumerate(best_runs[:3], 1):
    run_id = run['run_id']
    has_model = check_model_artifact(run_id)
    
    status = "✅" if has_model else "❌"
    print(f"{status} Rank {i} (Run: {run_id[:16]}...): ", end="")
    
    if has_model:
        print("Has model artifact")
    else:
        print("NO model artifact!")
        issues_found.append(i)

if issues_found:
    print(f"\n⚠️  WARNING: {len(issues_found)} run(s) don't have model artifacts!")
    print(f"   Ranks affected: {issues_found}")
    print("\n💡 This usually happens with nested runs from Session 5.")
    print("   We need to fix this before registration!")
else:
    print("\n✅ All runs have model artifacts!")
    print("   You can skip the fix cells and go to registration.")

🔍 Checking if our best runs have model artifacts...

❌ Rank 1 (Run: 8cc92639f5374bd0...): NO model artifact!
❌ Rank 2 (Run: a0daa66b52184b36...): NO model artifact!
❌ Rank 3 (Run: 796bad712edf4a9b...): NO model artifact!

⚠️  WARNING: 3 run(s) don't have model artifacts!
   Ranks affected: [1, 2, 3]

💡 This usually happens with nested runs from Session 5.
   We need to fix this before registration!


# 🔧 Parent-Child Run Issue & Solution

## 🎯 The Problem

### What Happened in Session 5

In Session 5, you created **nested run structures** for hyperparameter tuning:

```python
# Session 5 Pattern
with mlflow.start_run(run_name="LogisticRegression_GridSearch") as parent_run:
    # Parent run metadata
    
    for params in param_grid:
        with mlflow.start_run(run_name=f"LR_C{C}_penalty{penalty}", nested=True):
            # Train model with these params
            # Log metrics and params
            # BUT: Don't log model artifact here!
    
    # After finding best params...
    # Train best model and log ONLY at parent level
    mlflow.sklearn.log_model(best_model, "model")  # Only in parent!
```

**Result:**
- ✅ **Parent run**: Has the BEST model artifact
- ❌ **Child runs**: Only have metrics and parameters (NO model artifacts)

### The Registration Problem

When you try to register models using child run IDs:

```python
# This FAILS if run_id is a child run!
mlflow.register_model(
    model_uri=f"runs:/{child_run_id}/model",  # ❌ No model here!
    name="MyModel"
)
```

**Error:**
```
OSError: No such file or directory: '.../artifacts/model/.'
```

**Why it fails:**
1. `session5_best_runs.json` contains child run IDs (best metrics)
2. Child runs don't have model artifacts
3. Model is only in parent run
4. Registration tries to access non-existent artifact

---

## ✅ The Solution

### Step 1: Understand Your Run Structure

Run the verification utility to see your run structure:

```bash
python utils/find_parent_runs.py --verify-run YOUR_RUN_ID
```

**Output shows:**
```
🔍 RUN STRUCTURE VERIFICATION
======================================================================

Run ID: a05a43ad49244330878cd7d21d75a567
Run Name: LR_C0.1_penalty_l2

⚠️  This is a CHILD run
   Parent ID: abc123def456...

📦 Artifacts:
   ❌ No artifacts found  ← THIS IS THE PROBLEM!

📊 Metrics:
   accuracy: 0.8750  ← Best metrics are here
   
⚙️  Parameters:
   C: 0.1
   penalty: l2
```

### Step 2: Generate Corrected JSON

The utility automatically finds parent runs with model artifacts:

```bash
python utils/find_parent_runs.py \
  --input-json session5_best_runs.json \
  --output-json session5_best_runs_corrected.json
```

**What it does:**
1. Reads your original `session5_best_runs.json`
2. For each run, checks if it has model artifact
3. If not, finds the parent run
4. Verifies parent has the model
5. Creates corrected JSON with parent run IDs

**Example output:**
```
Processing Rank 1:
======================================================================
Original run_id: a05a43ad4924...
⚠️  Run a05a43ad4924... is a child run (no artifact)
   Checking parent: abc123def456...
   ✅ Parent run has model artifact!
✅ Corrected to parent: abc123def456...

💾 Saving corrected runs to: session5_best_runs_corrected.json
✅ Saved 3 corrected runs

📊 Summary:
   Total runs: 3
   Corrected to parent: 3
   Already had artifacts: 0
```

### Step 3: Use Corrected JSON for Registration

Now use the corrected JSON file:

```bash
# Option 1: Batch register all top 3
python registry_workflows/register_session5_models.py \
  --json-path session5_best_runs_corrected.json

# Option 2: Register individual model
python registry_workflows/register_model.py \
  --use-session5-best \
  --rank 1 \
  --model-name "ChurnModel" \
  --json-path session5_best_runs_corrected.json
```

**Success!** ✅

---

## 📊 Corrected JSON Structure

The corrected JSON adds metadata about the correction:

```json
[
  {
    "rank": 1,
    "run_id": "abc123def456...",  ← Parent ID (has model!)
    "original_run_id": "a05a43ad4924...",  ← Child ID (best metrics)
    "corrected": true,
    "correction_reason": "Parent run contains model artifact",
    "experiment_name": "Hyperparameter_Tuning",
    "model_type": "LogisticRegression",
    "accuracy": 0.8750,
    "f1_score": 0.8723
  },
  ...
]
```

**Key additions:**
- `run_id`: Updated to parent (registrable)
- `original_run_id`: Preserved for reference
- `corrected`: Flag indicating correction
- `correction_reason`: Explanation

---

## 🎓 Understanding the Architecture

### Session 5 Nested Run Pattern

```
Parent Run (GridSearch)
├── run_id: abc123def456
├── Artifacts: model/ ← ONLY MODEL LOCATION!
├── Metrics: best_accuracy=0.8750
└── Tags: Best hyperparameters

    ├── Child Run 1 (C=0.1, penalty=l2)
    │   ├── run_id: child_001
    │   ├── Metrics: accuracy=0.8750 ← BEST METRICS!
    │   ├── Params: C=0.1, penalty=l2
    │   └── Artifacts: NONE
    │
    ├── Child Run 2 (C=1.0, penalty=l2)
    │   ├── run_id: child_002
    │   ├── Metrics: accuracy=0.8720
    │   ├── Params: C=1.0, penalty=l2
    │   └── Artifacts: NONE
    │
    └── Child Run 3 (C=10.0, penalty=l1)
        ├── run_id: child_003
        ├── Metrics: accuracy=0.8650
        ├── Params: C=10.0, penalty=l1
        └── Artifacts: NONE
```

**The Mismatch:**
- **Best metrics** are in Child Run 1
- **Best model artifact** is in Parent Run
- We need Parent Run ID for registration!

---

## 🔍 Manual Verification

If you want to verify manually:

### Option 1: MLflow UI

1. Open MLflow UI: `mlflow ui`
2. Find your child run with best metrics
3. Look for "Parent Run" link in UI
4. Click it to see parent run
5. Check "Artifacts" tab in parent
6. Verify `model/` folder exists
7. Use parent run ID for registration

### Option 2: Python Code

```python
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Your child run ID (with best metrics)
child_run_id = "a05a43ad49244330878cd7d21d75a567"

# Get parent run ID
child_run = client.get_run(child_run_id)
parent_run_id = child_run.data.tags.get("mlflow.parentRunId")

print(f"Child run: {child_run_id}")
print(f"Parent run: {parent_run_id}")

# Verify parent has model
artifacts = client.list_artifacts(parent_run_id, "model")
print(f"Model artifacts: {len(artifacts) > 0}")

# Use parent_run_id for registration!
```

---

## 💡 Best Practices Going Forward

### For Future Sessions

To avoid this issue in the future:

#### Option 1: Log Model in Each Child Run
```python
# In Session 5 pattern
for params in param_grid:
    with mlflow.start_run(nested=True):
        # Train model
        model = train(params)
        
        # Log model in EVERY run
        mlflow.sklearn.log_model(model, "model")  # ✅ Model in child!
        
        # Log metrics
        mlflow.log_metrics(metrics)
```

**Pros:** Each run is independently registrable  
**Cons:** More storage space (multiple model copies)

#### Option 2: Save Best Run ID Correctly
```python
# In Session 5, when saving best runs
best_results = []

with mlflow.start_run(run_name="GridSearch") as parent_run:
    # ... tuning logic ...
    
    # Save parent run ID (which has model)
    best_results.append({
        'rank': 1,
        'run_id': parent_run.info.run_id,  # ✅ Parent ID!
        'child_run_id': best_child_run_id,  # For reference
        'metrics': best_metrics
    })
```

**Pros:** Correct from the start  
**Cons:** Need to track both parent and child

#### Option 3: Use Current Solution
```python
# Generate best runs as usual in Session 5
# Then in Session 6, run correction utility
python utils/find_parent_runs.py
```

**Pros:** Works with existing Session 5 code  
**Cons:** Extra correction step  
**Recommended:** ✅ For current course!

---

## 🚀 Quick Start Guide

### If You're Stuck Now

1. **Check if you have the issue:**
   ```bash
   python utils/find_parent_runs.py --verify-run YOUR_RUN_ID
   ```

2. **Generate corrected JSON:**
   ```bash
   python utils/find_parent_runs.py
   ```

3. **Register models:**
   ```bash
   python registry_workflows/register_session5_models.py \
     --json-path session5_best_runs_corrected.json
   ```

4. **Done!** ✅

---

## ❓ FAQs

### Q: Why did Session 5 use this pattern?

**A:** To save storage space and follow hyperparameter tuning best practices:
- Only save the best model
- Keep all metrics for comparison
- Parent-child structure keeps experiments organized

### Q: Is this an MLflow bug?

**A:** No! This is expected behavior:
- MLflow correctly implements nested runs
- Model registration requires model artifacts
- The mismatch is in our JSON generation logic

### Q: Will this affect my Session 5 work?

**A:** No! Your Session 5 experiments are fine:
- All runs are properly logged
- All metrics are preserved
- Models are safely stored in parent runs
- Just need correct run IDs for registration

### Q: Should I redo Session 5?

**A:** Absolutely not! Just run the correction utility:
```bash
python utils/find_parent_runs.py
```

### Q: What if I can't find parent runs?

**A:** This means:
1. Your runs might not have nested structure
2. Or models were logged differently
3. Check with: `python utils/find_parent_runs.py --verify-run YOUR_ID`
4. You may need to re-train and log models properly

---

## 📚 Additional Resources

### Related Files
- `utils/find_parent_runs.py` - Correction utility
- `registry_workflows/register_session5_models.py` - Updated registration script
- `session5_best_runs.json` - Original (may have child IDs)
- `session5_best_runs_corrected.json` - Fixed (parent IDs)

### Documentation
- MLflow Nested Runs: https://mlflow.org/docs/latest/tracking.html#organizing-runs-in-experiments
- Model Registry: https://mlflow.org/docs/latest/model-registry.html

---

## ✅ Summary

**The Issue:** Child runs have best metrics but no model artifacts  
**The Solution:** Use parent run IDs which have the models  
**The Tool:** `utils/find_parent_runs.py` automatically fixes this  
**The Result:** Successful model registration! 🎉

**This is a real-world MLOps scenario - you just learned to handle it! 💪**


In [6]:
"""
🔧 Running Fix Utility

This will automatically find parent runs with model artifacts
and generate a corrected JSON file.
"""

print("=" * 70)
print("🔧 FIXING PARENT-CHILD RUN ISSUE")
print("=" * 70)

# Run the fix utility
!python utils/find_parent_runs.py --input-json session5_best_runs.json --output-json session5_best_runs_corrected.json

print("\n" + "=" * 70)
print("✅ FIX UTILITY COMPLETE!")
print("=" * 70)

# Verify the corrected file was created
if Path('session5_best_runs_corrected.json').exists():
    print("\n✅ Created: session5_best_runs_corrected.json")
    print("   This file now contains parent run IDs with model artifacts!")
else:
    print("\n❌ Error: Corrected file not created")
    print("   Please check the error messages above")

🔧 FIXING PARENT-CHILD RUN ISSUE
🔧 CORRECTING SESSION 5 BEST RUNS FOR MODEL REGISTRATION

📁 Will check for models in paths: ['model', 'best_model']

📂 Loading: session5_best_runs.json
   Found 3 runs

Processing Rank 1:
Original run_id: 8cc92639f5374bd0...
⚠️  Run 8cc92639f5374bd0... is a child run (no artifact)
   Checking parent: d05738eb9bbe44f6...
   ✅ Parent run has model artifact at: best_model/
✅ Corrected to parent: d05738eb9bbe44f6...
   Found model at: best_model/

Processing Rank 2:
Original run_id: a0daa66b52184b36...
⚠️  Run a0daa66b52184b36... is a child run (no artifact)
   Checking parent: d05738eb9bbe44f6...
   ✅ Parent run has model artifact at: best_model/
✅ Corrected to parent: d05738eb9bbe44f6...
   Found model at: best_model/

Processing Rank 3:
Original run_id: 796bad712edf4a9b...
⚠️  Run 796bad712edf4a9b... is a child run (no artifact)
   Checking parent: d05738eb9bbe44f6...
   ✅ Parent run has model artifact at: best_model/
✅ Corrected to parent: d05738eb9bbe44f

In [7]:
"""
🔍 Verifying the Fix

Let's compare what changed between original and corrected JSON
"""

print("📊 Comparing Original vs Corrected JSON\n")
print("=" * 70)

# Load both files
with open('session5_best_runs.json') as f:
    original_runs = json.load(f)

original_runs = original_runs['top_runs']

with open('session5_best_runs_corrected.json') as f:
    corrected_runs = json.load(f)

# Compare each rank
for i in range(min(3, len(original_runs))):
    orig = original_runs[i]
    corr = corrected_runs[i]
    
    print(f"\nRank {i+1}:")
    print("-" * 70)
    print(f"Original run_id:  {orig['run_id'][:16]}...")
    print(f"Corrected run_id: {corr['run_id'][:16]}...")
    
    if corr.get('corrected'):
        print(f"✅ Changed: Child → Parent run")
        print(f"   (Model now accessible!)")
    else:
        print(f"✅ No change needed (already had model)")
    
    # Verify model exists in corrected run
    has_model = check_model_artifact(corr['run_id'])
    print(f"Model artifact: {'✅ Present' if has_model else '❌ Missing'}")

print("\n" + "=" * 70)
print("✅ All corrected runs now have model artifacts!")
print("=" * 70)

📊 Comparing Original vs Corrected JSON


Rank 1:
----------------------------------------------------------------------
Original run_id:  8cc92639f5374bd0...
Corrected run_id: d05738eb9bbe44f6...
✅ Changed: Child → Parent run
   (Model now accessible!)
Model artifact: ✅ Present

Rank 2:
----------------------------------------------------------------------
Original run_id:  a0daa66b52184b36...
Corrected run_id: d05738eb9bbe44f6...
✅ Changed: Child → Parent run
   (Model now accessible!)
Model artifact: ✅ Present

Rank 3:
----------------------------------------------------------------------
Original run_id:  796bad712edf4a9b...
Corrected run_id: d05738eb9bbe44f6...
✅ Changed: Child → Parent run
   (Model now accessible!)
Model artifact: ✅ Present

✅ All corrected runs now have model artifacts!


---

## Section 4: Registering Models (20 min)

### Method 1: Register via Python API

Let's register your best Session 5 model!

In [8]:
"""
📦 Registering Our First Model

Now that we have corrected run IDs, let's register our best model!
"""

# Use CORRECTED JSON
with open('session5_best_runs_corrected.json') as f:
    corrected_runs = json.load(f)

best_run = corrected_runs[0]

print("📦 Registering Best Model from Session 5\n")
print(f"Model Details:")
print(f"  - Rank: {best_run['rank']}")
print(f"  - Run ID: {best_run['run_id'][:16]}...")
# print(f"  - Model Type: {best_run['model_type']}")
print(f"  - Accuracy: {best_run['accuracy']:.4f}")

# Register the model
model_name = "CustomerChurnClassifier"
model_uri = f"runs:/{best_run['run_id']}/best_model"

print(f"\n🔄 Registering as: {model_name}")

result = mlflow.register_model(
    model_uri=model_uri,
    name=model_name
)

print(f"\n✅ Model registered successfully!")
print(f"   - Name: {result.name}")
print(f"   - Version: {result.version}")
print(f"   - Status: {result.status}")

# Add description
description = f"""
Best model from Session 5 (Rank {best_run['rank']})
Accuracy: {best_run['accuracy']:.4f}
"""

client.update_model_version(
    name=model_name,
    version=result.version,
    description=description
)

print(f"\n📝 Added description and metadata")

📦 Registering Best Model from Session 5

Model Details:
  - Rank: 1
  - Run ID: d05738eb9bbe44f6...
  - Accuracy: 0.6835

🔄 Registering as: CustomerChurnClassifier

✅ Model registered successfully!
   - Name: CustomerChurnClassifier
   - Version: 1
   - Status: READY

📝 Added description and metadata


Successfully registered model 'CustomerChurnClassifier'.
Created version '1' of model 'CustomerChurnClassifier'.


### What Just Happened?

When you register a model:

1. **Model gets copied** to Registry storage
2. **Version number assigned** automatically (starts at 1)
3. **Lineage preserved** - links back to training run
4. **Stage set to "None"** - not yet validated
5. **Metadata stored** in registry database

**Pro Tip:** You can register the same model name multiple times - each registration creates a new version!

### Adding Rich Metadata

Let's add helpful information to our registered model:

In [ ]:
print("Adding metadata to registered model...\n")

# Get model details
model_version = result.version
accuracy = best_runs[0]['accuracy']

# Add description
description = f"""
Customer Churn Prediction Model - Version {model_version}

**Training Details:**
- Trained on Session 5 data
- Algorithm: Random Forest Classifier
- Accuracy: {accuracy:.4f}
- Training Date: 2025-Q4
- Dataset: customer_churn.csv (10,000 samples)

**Model Characteristics:**
- Purpose: Predict customer churn probability
- Input: Customer demographic and usage features
- Output: Binary classification (0 = No Churn, 1 = Churn)

**Validation:**
- Cross-validated on Session 5 test set
- Evaluated for bias and fairness
- Meets performance threshold (>85% accuracy)
"""

client.update_model_version(
    name=model_name,
    version=model_version,
    description=description
)

print(f"Description added!")
print(f"\nDescription preview:")
print(description[:200] + "...")

Adding metadata to registered model...

Description added!

Description preview:

Customer Churn Prediction Model - Version 1

**Training Details:**
- Trained on Session 5 data
- Algorithm: Random Forest Classifier
- Accuracy: 0.6835
- Training Date: 2024-Q4
- Dataset: customer_ch...


### Adding Tags for Governance

In [ ]:
print("Adding governance tags...\n")

# Add tags
tags = {
    "algorithm": "LogisticRegression",
    "accuracy": f"{accuracy:.4f}",
    "training_data": "customer_churn_2025Q4",
    "trained_by": "data_science_team",
    "framework": "scikit-learn",
    "session": "session_5",
    "validation_status": "pending"
}

for key, value in tags.items():
    client.set_model_version_tag(
        name=model_name,
        version=model_version,
        key=key,
        value=value
    )
    print(f"   {key}: {value}")

print(f"\nAll tags added to version {model_version}!")

Adding governance tags...

   algorithm: LogisticRegression
   accuracy: 0.6835
   training_data: customer_churn_2024Q4
   trained_by: data_science_team
   framework: scikit-learn
   session: session_5
   validation_status: pending

All tags added to version 1!


### Registering Multiple Models

Let's register your top 3 Session 5 models!

In [11]:
print("Registering top 3 Session 5 models...\n")

registered_models = []

for i, run_info in enumerate(corrected_runs, 1):
    print(f"\n{i}. Registering Rank {run_info['rank']} model...")
    
    model_name_rank = f"ChurnModel_Rank{i}"
    model_uri = f"runs:/{run_info['run_id']}/best_model"
    
    result = mlflow.register_model(
        model_uri=model_uri,
        name=model_name_rank
    )
    
    # Add description
    desc = f"Rank {i} model from Session 5. Accuracy: {run_info['accuracy']:.4f}"
    client.update_model_version(
        name=model_name_rank,
        version=result.version,
        description=desc
    )
    
    registered_models.append({
        'name': model_name_rank,
        'version': result.version,
        'accuracy': run_info['accuracy']
    })
    
    print(f"   {model_name_rank} v{result.version} registered")

print(f"\n" + "="*70)
print("ALL MODELS REGISTERED!")
print("="*70)

print(f"\nRegistration Summary:")
for model in registered_models:
    print(f"   - {model['name']} v{model['version']} (Acc: {model['accuracy']:.4f})")

print(f"\nView in MLflow UI:")
print(f"   1. mlflow ui")
print(f"   2. Click 'Models' tab")
print(f"   3. See all 3 registered models!")

Registering top 3 Session 5 models...


1. Registering Rank 1 model...
   ChurnModel_Rank1 v1 registered

2. Registering Rank 2 model...
   ChurnModel_Rank2 v1 registered

3. Registering Rank 3 model...
   ChurnModel_Rank3 v1 registered

ALL MODELS REGISTERED!

Registration Summary:
   - ChurnModel_Rank1 v1 (Acc: 0.6835)
   - ChurnModel_Rank2 v1 (Acc: 0.6830)
   - ChurnModel_Rank3 v1 (Acc: 0.6820)

View in MLflow UI:
   1. mlflow ui
   2. Click 'Models' tab
   3. See all 3 registered models!


Successfully registered model 'ChurnModel_Rank1'.
Created version '1' of model 'ChurnModel_Rank1'.
Successfully registered model 'ChurnModel_Rank2'.
Created version '1' of model 'ChurnModel_Rank2'.
Successfully registered model 'ChurnModel_Rank3'.
Created version '1' of model 'ChurnModel_Rank3'.


### Viewing Registered Models

Let's query the registry to see what we have:

In [12]:
# List all registered models
print("Registered Models in Registry:\n")

registered_models_list = client.search_registered_models()

if not registered_models_list:
    print("   (No models registered yet)")
else:
    for rm in registered_models_list:
        print(f"Model: {rm.name}")
        print(f"   - Latest Versions: {len(rm.latest_versions)}")
        
        for version in rm.latest_versions:
            print(f"   - Version {version.version}: {version.current_stage}")
        print()

print(f"\nTotal registered models: {len(registered_models_list)}")

Registered Models in Registry:

Model: ChurnModel_Rank1
   - Latest Versions: 1
   - Version 1: None

Model: ChurnModel_Rank2
   - Latest Versions: 1
   - Version 1: None

Model: ChurnModel_Rank3
   - Latest Versions: 1
   - Version 1: None

Model: CustomerChurnClassifier
   - Latest Versions: 1
   - Version 1: None


Total registered models: 4


---

## Section 4: Managing Model Lifecycle (25 min)

### Understanding Stage Transitions

Now that models are registered, let's move them through the lifecycle:

### Stage Transition Workflow

**Recommended Production Workflow:**

```
1. Register -> Stage: None
   - Model just registered
   - Not yet validated
   - Action: Review model metrics

2. Validate -> Stage: Staging  
   - Run offline validation tests
   - Compare with current production
   - Action: Shadow test or A/B test

3. Approve -> Stage: Production
   - Staging validation passed
   - Stakeholder approval received
   - Action: Deploy to production

4. Retire -> Stage: Archived
   - Model replaced by newer version
   - Keep for audit trail
   - Action: Document retirement reason
```

### Promoting to Staging

Let's promote our best model to Staging for testing:

In [ ]:
print("Promoting model to Staging...\n")

model_name = "CustomerChurnClassifier"
version = 1

print(f"Model: {model_name}")
print(f"Version: {version}")
print(f"Target Stage: Staging")

# Transition to Staging
client.transition_model_version_stage(
    name=model_name,
    version=version,
    stage="Staging"
)

print(f"\nModel promoted to Staging!")

# Add promotion metadata
from datetime import datetime

client.set_model_version_tag(
    name=model_name,
    version=version,
    key="promoted_to_staging_at",
    value=datetime.now().isoformat()
)

client.set_model_version_tag(
    name=model_name,
    version=version,
    key="promoted_by",
    value="data_science_team"
)

print(f"\nPromotion metadata added")
print(f"\nNext Steps:")
print(f"   1. Run validation tests on Staging model")
print(f"   2. Compare with Production (if exists)")
print(f"   3. Get stakeholder approval")
print(f"   4. Promote to Production")

Promoting model to Staging...

Model: CustomerChurnClassifier
Version: 1
Target Stage: Staging

Model promoted to Staging!

Promotion metadata added

Next Steps:
   1. Run validation tests on Staging model
   2. Compare with Production (if exists)
   3. Get stakeholder approval
   4. Promote to Production


C:\Users\Supriya\AppData\Local\Temp\ipykernel_27704\3495626108.py:11: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/2.9.2/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


### Loading Model from Staging

Let's verify we can load and use the Staging model:

In [15]:
print("Loading model from Staging stage...\n")

# Load by stage (not by version!)
model_uri = f"models:/{model_name}/Staging"
print(f"Model URI: {model_uri}")

import mlflow.pyfunc
model = mlflow.pyfunc.load_model(model_uri)

print(f"\nModel loaded successfully!")
print(f"\nModel Info:")
print(f"   - Model: {model_name}")
print(f"   - Stage: Staging")
print(f"   - Ready for predictions!")
print(f"   - Type: {type(model)}")
print(f"   - Flavor: {model.metadata.flavors}")

print(f"\nKey Insight:")
print(f"   Loading by STAGE (not version) means:")
print(f"   - Always get the current Staging model")
print(f"   - No code changes when you promote new versions")
print(f"   - Production code stays the same!")

Loading model from Staging stage...

Model URI: models:/CustomerChurnClassifier/Staging


c:\SkillfyME\MLOps_with_Agentic_AI\Module-1_ Python_and_MLOps_Foundations\session_5+6_handson\.venv\Lib\site-packages\mlflow\store\artifact\utils\models.py:32: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/2.9.2/model-registry.html#migrating-from-stages
  latest = client.get_latest_versions(name, None if stage is None else [stage])



Model loaded successfully!

Model Info:
   - Model: CustomerChurnClassifier
   - Stage: Staging
   - Ready for predictions!
   - Type: <class 'mlflow.pyfunc.PyFuncModel'>
   - Flavor: {'python_function': {'env': {'conda': 'conda.yaml', 'virtualenv': 'python_env.yaml'}, 'loader_module': 'mlflow.sklearn', 'model_path': 'model.pkl', 'predict_fn': 'predict', 'python_version': '3.11.9'}, 'sklearn': {'code': None, 'pickled_model': 'model.pkl', 'serialization_format': 'cloudpickle', 'sklearn_version': '1.3.2'}}

Key Insight:
   Loading by STAGE (not version) means:
   - Always get the current Staging model
   - No code changes when you promote new versions
   - Production code stays the same!


In [16]:
# Test prediction
print(f"\n🎯 Testing prediction...")
import numpy as np

# Create sample data
sample_data = np.random.rand(1, 12)  # Adjust shape to your model
try:
    prediction = model.predict(sample_data)
    print(f"✅ Prediction successful: {prediction}")
    print(f"\n🎉 Model is fully functional and ready for production!")
except Exception as e:
    print(f"⚠️  Prediction error: {e}")
    print(f"   (This might be due to feature mismatch, not the model itself)")

print("\n" + "=" * 70)
print("✅ MODEL SUCCESSFULLY LOADED AND TESTED!")
print("=" * 70)


🎯 Testing prediction...
✅ Prediction successful: [0]

🎉 Model is fully functional and ready for production!

✅ MODEL SUCCESSFULLY LOADED AND TESTED!


c:\SkillfyME\MLOps_with_Agentic_AI\Module-1_ Python_and_MLOps_Foundations\session_5+6_handson\.venv\Lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


### Promoting to Production

After validation, let's promote to Production!

In [ ]:
print("Promoting model to Production...\n")

# Simulate validation checks
print("Validation Checklist:")
print("   [x] Offline metrics meet threshold")
print("   [x] Staging tests passed")
print("   [x] Stakeholder approval received")
print("   [x] Documentation updated")

print(f"\nPromoting to Production...")

# Promote to Production
client.transition_model_version_stage(
    name=model_name,
    version=version, # later to version = 2
    stage="Production",
    archive_existing_versions=False  # Don't archive yet (no existing prod model)
)

print(f"\nModel is now in PRODUCTION!")

# Add production metadata
client.set_model_version_tag(
    name=model_name,
    version=version,
    key="promoted_to_production_at",
    value=datetime.now().isoformat()
)

client.set_model_version_tag(
    name=model_name,
    version=version,
    key="production_approved_by",
    value="ml_engineering_lead"
)

print(f"\nIMPORTANT:")
print(f"   This model is now serving live predictions!")
print(f"   Make sure to:")
print(f"   1. Monitor model performance closely")
print(f"   2. Set up alerts for degradation")
print(f"   3. Have rollback plan ready")
print(f"   4. Document deployment in runbook")

Promoting model to Production...

Validation Checklist:
   [x] Offline metrics meet threshold
   [x] Staging tests passed
   [x] Stakeholder approval received
   [x] Documentation updated

Promoting to Production...

Model is now in PRODUCTION!

IMPORTANT:
   This model is now serving live predictions!
   Make sure to:
   1. Monitor model performance closely
   2. Set up alerts for degradation
   3. Have rollback plan ready
   4. Document deployment in runbook


C:\Users\Supriya\AppData\Local\Temp\ipykernel_27704\1801967976.py:13: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/2.9.2/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


---

## Section 5: Comparing Model Versions (15 min)

### Why Compare Versions?

Before promoting a new model, you should compare it with the current production model!

In [18]:
print("Comparing Model Versions...\n")

# Let's register a second version to compare
print("Step 1: Registering a new model version...")

# Use the rank 2 model from Session 5
second_best_run = best_runs[1]
model_uri_v2 = f"runs:/{second_best_run['run_id']}/best_model"

result_v2 = mlflow.register_model(
    model_uri=model_uri_v2,
    name=model_name  # Same model name = new version
)

print(f"   Version {result_v2.version} registered")

# Promote v2 to Staging
print(f"\nStep 2: Promoting v{result_v2.version} to Staging...")
client.transition_model_version_stage(
    name=model_name,
    version=result_v2.version,
    stage="Staging"
)
print(f"   Version {result_v2.version} in Staging")

print(f"\nNow we have:")
print(f"   - Version {version}: Production")
print(f"   - Version {result_v2.version}: Staging")
print(f"\n   Let's compare them!")

Comparing Model Versions...

Step 1: Registering a new model version...
   Version 2 registered

Step 2: Promoting v2 to Staging...
   Version 2 in Staging

Now we have:
   - Version 1: Production
   - Version 2: Staging

   Let's compare them!


Registered model 'CustomerChurnClassifier' already exists. Creating a new version of this model...
Created version '2' of model 'CustomerChurnClassifier'.
C:\Users\Supriya\AppData\Local\Temp\ipykernel_27704\2161419305.py:19: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/2.9.2/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


### Side-by-Side Comparison

In [20]:
def compare_model_versions(model_name, version1, version2):
    """
    Compare two model versions side-by-side
    """
    print(f"\n{'='*70}")
    print(f"COMPARING: {model_name} v{version1} vs v{version2}")
    print(f"{'='*70}\n")
    
    # Get both versions
    v1 = client.get_model_version(model_name, version1)
    v2 = client.get_model_version(model_name, version2)
    
    # Basic info
    print(f"Basic Information:")
    print(f"   Version {version1}:")
    print(f"      - Stage: {v1.current_stage}")
    print(f"      - Status: {v1.status}")
    
    print(f"   Version {version2}:")
    print(f"      - Stage: {v2.current_stage}")
    print(f"      - Status: {v2.status}")
    
    # Get metrics from runs
    run1 = client.get_run(v1.run_id)
    run2 = client.get_run(v2.run_id)
   
    metrics1 = run1.data.metrics
    metrics2 = run2.data.metrics
   
    # Compare metrics
    print(f"\nPerformance Metrics:")
    print(f"\n   {'Metric':<15} {'Version '+str(version1):<15} {'Version '+str(version2):<15} {'Winner':<10}")
    print(f"   {'-'*60}")
    
    for metric in ['test_accuracy', 'precision', 'recall', 'f1_score', 'roc_auc']:
        metric_1 = metric
        metric_2 = metric
        if metric == 'test_accuracy':
            if 'best_test_accuracy' in metrics1:
                metric_1 = 'best_test_accuracy'
            elif 'test_accuracy' in metrics1:
                metric_1 = 'test_accuracy'
            else:
                metric_1 = metric
            
            if 'best_test_accuracy' in metrics2:
                metric_2 = 'best_test_accuracy'
            elif 'test_accuracy' in metrics2:
                metric_2 = 'test_accuracy'
            else:
                metric_2 = metric

        if metric_1 in metrics1 and metric_2 in metrics2:
            m1 = metrics1[metric_1]
            m2 = metrics2[metric_2]
          
            winner = "V1" if m1 > m2 else "V2" if m2 > m1 else "Tie"
            
            print(f"   {metric:<15} {m1:<15.4f} {m2:<15.4f} {winner:<10}")
    
    # Overall recommendation
    print(f"\nRecommendation:")
    
    acc1 = metrics1.get('test_accuracy', metrics1.get('best_test_accuracy', 0))
    acc2 = metrics2.get('test_accuracy', metrics1.get('best_test_accuracy', 0))
    
    if acc2 > acc1:
        diff = ((acc2 - acc1) / acc1) * 100
        print(f"   Version {version2} performs {diff:.2f}% better")
        print(f"   Recommend promoting Version {version2} to Production")
    elif acc1 > acc2:
        diff = ((acc1 - acc2) / acc2) * 100
        print(f"   Version {version1} performs {diff:.2f}% better")
        print(f"   Keep Version {version1} in Production")
    else:
        print(f"   Similar performance - consider other factors")
    
    print(f"\n{'='*70}\n")

# Run comparison
compare_model_versions(model_name, version, result_v2.version)


COMPARING: CustomerChurnClassifier v1 vs v2

Basic Information:
   Version 1:
      - Stage: Production
      - Status: READY
   Version 2:
      - Stage: Staging
      - Status: READY

Performance Metrics:

   Metric          Version 1       Version 2       Winner    
   ------------------------------------------------------------
   test_accuracy   0.6835          0.6830          V1        

Recommendation:
   Version 1 performs 0.07% better
   Keep Version 1 in Production




---

## Section 6: Rollback Scenarios (10 min)

### When Things Go Wrong in Production

Let's simulate a production incident and rollback:

### Production Incident Simulation

**Scenario:**
- Your latest model (v3) was promoted to Production
- After 2 hours, performance is degrading
- Accuracy dropped from 87.5% to 72%
- False positives increased by 300%
- Need to rollback IMMEDIATELY!

**Action:** Roll back to the previous stable version

In [23]:
print("PRODUCTION INCIDENT DETECTED!\n")
print("Current Production Model: Version 2")
print("Issue: Performance degradation")
print("Status: CRITICAL\n")

print("Initiating Rollback Procedure...\n")

# Step 1: Identify last known good version
print("Step 1: Identifying last stable version...")
archived_versions = client.get_latest_versions(model_name, stages=["Archived"])

if not archived_versions:
    # For demo, we'll use version 1
    last_good_version = 1
    print(f"   Found: Version {last_good_version}")
else:
    last_good_version = archived_versions[0].version
    print(f"   Found: Version {last_good_version}")

# Step 2: Get current production version
print(f"\nStep 2: Archiving failing production model...")
current_prod = client.get_latest_versions(model_name, stages=["Production"])[0]

client.transition_model_version_stage(
    name=model_name,
    version=current_prod.version,
    stage="Archived"
)

# Add rollback reason
client.set_model_version_tag(
    name=model_name,
    version=current_prod.version,
    key="archived_reason",
    value="Production performance degradation - emergency rollback"
)

print(f"   Version {current_prod.version} archived")

# Step 3: Promote last good version back to Production
print(f"\nStep 3: Restoring Version {last_good_version} to Production...")

client.transition_model_version_stage(
    name=model_name,
    version=last_good_version,
    stage="Production"
)

# Log rollback metadata
client.set_model_version_tag(
    name=model_name,
    version=last_good_version,
    key="rollback_from_version",
    value=str(current_prod.version)
)

client.set_model_version_tag(
    name=model_name,
    version=last_good_version,
    key="rollback_at",
    value=datetime.now().isoformat()
)

print(f"   Version {last_good_version} restored to Production")

print(f"\n" + "="*70)
print("ROLLBACK COMPLETE!")
print("="*70)

print(f"\nCurrent Status:")
print(f"   - Production: Version {last_good_version} (STABLE)")
print(f"   - Archived: Version {current_prod.version} (FAILED)")

print(f"\nPost-Rollback Actions:")
print(f"   1. Monitor production traffic closely")
print(f"   2. Verify performance has stabilized")
print(f"   3. Investigate root cause of failure")
print(f"   4. Update incident report")
print(f"   5. Notify stakeholders")

PRODUCTION INCIDENT DETECTED!

Current Production Model: Version 2
Issue: Performance degradation
Status: CRITICAL

Initiating Rollback Procedure...

Step 1: Identifying last stable version...
   Found: Version 1

Step 2: Archiving failing production model...
   Version 2 archived

Step 3: Restoring Version 1 to Production...
   Version 1 restored to Production

ROLLBACK COMPLETE!

Current Status:
   - Production: Version 1 (STABLE)
   - Archived: Version 2 (FAILED)

Post-Rollback Actions:
   1. Monitor production traffic closely
   2. Verify performance has stabilized
   3. Investigate root cause of failure
   4. Update incident report
   5. Notify stakeholders


C:\Users\Supriya\AppData\Local\Temp\ipykernel_27704\1907631123.py:10: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/2.9.2/model-registry.html#migrating-from-stages
  archived_versions = client.get_latest_versions(model_name, stages=["Archived"])
C:\Users\Supriya\AppData\Local\Temp\ipykernel_27704\1907631123.py:22: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/2.9.2/model-registry.html#migrating-from-stages
  current_prod = client.get_latest_versions(model_name, stages=["Production"])[0]
C:\Users\Supriya\AppData\Local\Tem

### Rollback Best Practices

**Before Rollback:**
- Verify the issue is model-related (not data/infrastructure)
- Identify symptoms and severity
- Locate last known good version
- Notify stakeholders

**During Rollback:**
- Act quickly but deliberately
- Document every step
- Tag rollback reason
- Monitor closely

**After Rollback:**
- Root cause analysis
- Postmortem report
- Implement preventive measures
- Update runbooks

**Pro Tip:** Practice rollbacks regularly so your team is prepared!

---

## Section 7: Approval Workflows (10 min)

### Implementing Approval Gates

Production models should require approval before deployment:

In [24]:
def request_approval(model_name, version, requester, justification):
    """
    Simulate approval request workflow
    """
    print(f"\n{'='*70}")
    print(f"PRODUCTION APPROVAL REQUEST")
    print(f"{'='*70}\n")
    
    print(f"Request Details:")
    print(f"   Model: {model_name}")
    print(f"   Version: {version}")
    print(f"   Requested by: {requester}")
    print(f"   Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    
    print(f"\nJustification:")
    print(f"   {justification}")
    
    # Get model metrics
    mv = client.get_model_version(model_name, version)
    run = client.get_run(mv.run_id)
    
    print(f"\nModel Performance:")
    for metric, value in sorted(run.data.metrics.items())[:5]:
        print(f"   - {metric}: {value:.4f}")
    
    # Validation checklist
    print(f"\nPre-Approval Checklist:")
    checklist = [
        "Offline metrics meet threshold (>85% accuracy)",
        "Staging validation passed",
        "Model documentation complete",
        "Rollback plan documented",
        "Monitoring dashboards set up",
        "Stakeholder notification sent"
    ]
    
    for item in checklist:
        print(f"   [x] {item}")
    
    # Tag for approval
    client.set_model_version_tag(
        name=model_name,
        version=version,
        key="approval_requested_at",
        value=datetime.now().isoformat()
    )
    
    client.set_model_version_tag(
        name=model_name,
        version=version,
        key="approval_requested_by",
        value=requester
    )
    
    print(f"\nApproval request sent to: ml_engineering_lead@company.com")
    print(f"\nStatus: PENDING APPROVAL")
    print(f"\n{'='*70}\n")

# Request approval for Staging model
request_approval(
    model_name="CustomerChurnClassifier",
    version=2,
    requester="data_scientist_arun",
    justification="New model shows 2.3% improvement in accuracy. Staging validation shows stable performance."
)


PRODUCTION APPROVAL REQUEST

Request Details:
   Model: CustomerChurnClassifier
   Version: 2
   Requested by: data_scientist_arun
   Date: 2025-11-08 18:48

Justification:
   New model shows 2.3% improvement in accuracy. Staging validation shows stable performance.

Model Performance:
   - test_accuracy: 0.6830

Pre-Approval Checklist:
   [x] Offline metrics meet threshold (>85% accuracy)
   [x] Staging validation passed
   [x] Model documentation complete
   [x] Rollback plan documented
   [x] Monitoring dashboards set up
   [x] Stakeholder notification sent

Approval request sent to: ml_engineering_lead@company.com

Status: PENDING APPROVAL




In [ ]:
def approve_model(model_name, version, approver, decision, notes):
    """
    Simulate approval decision
    """
    print(f"\n{'='*70}")
    print(f"APPROVAL DECISION")
    print(f"{'='*70}\n")
    
    print(f"Approver: {approver}")
    print(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    
    if decision == "approved":
        print(f"\nDECISION: APPROVED")
        
        # Add approval tags
        client.set_model_version_tag(
            name=model_name,
            version=version,
            key="approval_status",
            value="approved"
        )
        
        client.set_model_version_tag(
            name=model_name,
            version=version,
            key="approved_by",
            value=approver
        )
        
        client.set_model_version_tag(
            name=model_name,
            version=version,
            key="approved_at",
            value=datetime.now().isoformat()
        )
        
        print(f"\nNotes: {notes}")
        print(f"\nCleared for Production deployment!")
    else:
        print(f"\nDECISION: REJECTED")
        print(f"\nRejection Reason: {notes}")
    
    print(f"\n{'='*70}\n")

# Approve the model
approve_model(
    model_name="CustomerChurnClassifier",
    version=2,
    approver="ml_engineering_lead",
    decision="approved",
    notes="Model meets all criteria. Performance improvement validated. Approved for production deployment."
)

### Approval Workflow Summary

**Typical Enterprise Workflow:**

```
1. Data Scientist:
   - Trains and validates model
   - Registers in MLflow
   - Promotes to Staging
   - Requests production approval

2. ML Engineering Lead:
   - Reviews model metrics
   - Validates staging performance
   - Checks compliance requirements
   - Approves or rejects

3. MLOps/DevOps:
   - Promotes approved model to Production
   - Sets up monitoring
   - Documents deployment

4. Ongoing:
   - Monitor production performance
   - Trigger rollback if needed
   - Maintain audit trail
```

**Pro Tip:** Use MLflow tags to implement custom approval workflows!

---

## Section 8: Production Inference Patterns (10 min)

### Loading Models for Production

Let's see how to load and use models in production code:

In [ ]:
print("Production Inference Pattern\n")

# Load production model by stage (NOT by version!)
model_uri = f"models:/{model_name}/Production"
print(f"Loading: {model_uri}")

prod_model = mlflow.pyfunc.load_model(model_uri)
print(f"Production model loaded\n")

# Simulate prediction request
print("Sample Prediction Request:")

# Load test data
from utils.data_loader import load_customer_churn_data
_, X_test, _, _ = load_customer_churn_data()

# Make prediction on first sample
sample = X_test.iloc[0:1]

print(f"\nInput Features:")
print(sample.to_dict('records')[0])

# Predict
prediction = prod_model.predict(sample)

print(f"\nPrediction: {prediction[0]}")
print(f"   {'Churn' if prediction[0] == 1 else 'No Churn'}")

print(f"\nKey Insight:")
print(f"   By loading models by STAGE (not version):")
print(f"   - Your code never changes")
print(f"   - Just promote new model to Production")
print(f"   - Next request automatically uses new model!")

### Production API Pattern

Here's how you'd integrate with a FastAPI or Flask application:

In [ ]:
# Example production code pattern
production_code = '''
# production_api.py

from fastapi import FastAPI
import mlflow.pyfunc
import pandas as pd

app = FastAPI()

# Load production model at startup
MODEL_NAME = "CustomerChurnClassifier"
MODEL_STAGE = "Production"

model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}/{MODEL_STAGE}")

@app.post("/predict")
def predict(features: dict):
    # Convert to DataFrame
    input_df = pd.DataFrame([features])
    
    # Predict
    prediction = model.predict(input_df)[0]
    
    return {
        "prediction": int(prediction),
        "model": MODEL_NAME,
        "version": MODEL_STAGE,
        "churn_risk": "High" if prediction == 1 else "Low"
    }
'''

print("Production API Pattern:\n")
print(production_code)

print("\nBenefits:")
print("   - Load once at startup (fast)")
print("   - Load by stage (automatic updates)")
print("   - Model info endpoint (observability)")
print("   - Clean API interface")

---

## Section 9: Hands-On Exercises (15 min)

### Exercise 1: Complete Model Lifecycle

**Task:** Take your rank 3 Session 5 model through the complete lifecycle!

**Requirements:**
1. Register the model
2. Add description and tags
3. Promote to Staging
4. Request approval (simulate)
5. Approve (simulate)
6. Promote to Production
7. Verify you can load it

**Time:** 10 minutes

In [ ]:
# Exercise 1: Your turn!
print("Exercise 1: Complete Model Lifecycle\n")

# TODO: Get rank 3 model from best_runs
rank3_model = best_runs[2]
print(f"Working with Rank 3 model:")
print(f"   Run ID: {rank3_model['run_id'][:16]}...")
print(f"   Accuracy: {rank3_model['accuracy']:.4f}\n")

# Step 1: Register
print("Step 1: Register the model")
# TODO: Write code to register the model
# Hint: mlflow.register_model(...)

# Step 2: Add metadata
print("\nStep 2: Add description and tags")
# TODO: Add description and tags
# Hint: client.update_model_version(...)
# Hint: client.set_model_version_tag(...)

# Step 3: Promote to Staging
print("\nStep 3: Promote to Staging")
# TODO: Promote to Staging
# Hint: client.transition_model_version_stage(...)

# Step 4-6: Approval workflow
print("\nStep 4-6: Approval workflow")
# TODO: Request, approve, and promote to Production

# Step 7: Load and verify
print("\nStep 7: Load and verify")
# TODO: Load the production model and make a prediction

print("\nIf you completed all steps, you've mastered Model Registry!")

### Exercise 2: Rollback Simulation

**Task:** Practice rolling back from a failed deployment

**Scenario:**
- Your current production model (just deployed) is showing issues
- You need to rollback to the previous version
- Document the rollback reason

**Time:** 5 minutes

In [ ]:
# Exercise 2: Rollback simulation
print("Exercise 2: Rollback Simulation\n")

# TODO: Identify current production version
# TODO: Find previous archived version
# TODO: Archive current production
# TODO: Restore previous version to production
# TODO: Add rollback tags

print("\nHint: Use the rollback pattern from Section 6!")

---

## Part 2 Summary & Key Takeaways

### What You Learned in Part 2

**1. Model Registry Fundamentals**
- Registry is central model store for production
- Automatic versioning and lineage tracking
- Single source of truth for deployed models

**2. Model Lifecycle Management**
- Four stages: None -> Staging -> Production -> Archived
- Staged promotion workflow prevents accidents
- Always test in Staging before Production

**3. Registration & Metadata**
- Registered your Session 5 models
- Added descriptions, tags, and annotations
- Linked back to training runs
- Implemented governance practices

**4. Version Comparison**
- Compare metrics before promotion
- Make data-driven decisions
- Avoid deploying worse models

**5. Rollback Procedures**
- Quick recovery from production incidents
- Archive failed versions with reasons
- Maintain audit trail

**6. Approval Workflows**
- Gate production deployments
- Use tags for approval tracking
- Implement team review processes

**7. Production Patterns**
- Load models by stage (not version)
- Zero-downtime model updates
- Clean API integration

---

### Complete Session 6 Journey

**Part 1 (MLflow Projects):** Standardized your workflows  
**Part 2 (Model Registry):** Operationalized your models

**Combined Power:**
```
Session 5: Track experiments
    ↓
Part 1: Package as Projects
    ↓
Part 2: Register in Registry
    ↓
Session 7: Deploy to Production  --> NEXT!
```

---

### Best Practices Recap

**Registration:**
- Use descriptive model names
- Add rich metadata immediately
- Tag with algorithm, data version, etc.
- Document model purpose and limitations

**Promotion:**
- Always compare before promoting
- Test thoroughly in Staging
- Require approval for Production
- Archive old versions (don't delete)

**Production:**
- Load by stage (not version)
- Monitor performance continuously
- Have rollback plan ready
- Document all changes

**Governance:**
- Maintain audit trail
- Use tags for compliance
- Implement approval gates
- Regular reviews and cleanups

---

**Everything in the Registry is ready for deployment!**

---

## Session 6 Complete!

### What You Accomplished Today

**Part 1:**
- Understood MLflow Projects
- Created reusable ML workflows
- Converted Session 5 code to Projects
- Mastered entry points and parameters

**Part 2:**
- Registered Session 5 models in Registry
- Managed model lifecycle stages
- Compared and promoted models
- Practiced rollback procedures
- Implemented approval workflows

**You're now ready for production MLOps!**

---

### Practice Before Session 7

**Homework exercises:**

1. **Register all Session 5 models**
   - Register your remaining Session 5 models
   - Add rich metadata to each
   - Compare them side-by-side

2. **Create additional Projects**
   - Convert your Session 5 XGBoost code to a Project
   - Add multiple entry points
   - Share with teammates

3. **Practice workflows**
   - Simulate complete promotion workflow
   - Practice emergency rollback
   - Implement approval tags

4. **Explore MLflow UI**
   - Navigate Models tab thoroughly
   - Compare versions visually
   - Understand lineage graphs

---

### Key Takeaway

```
MLflow Projects + Model Registry = Production-Ready MLOps
```

**You now have:**
- Reproducible experiments (Projects)
- Managed model lifecycle (Registry)
- Production deployment foundation

**Session 7 will complete the pipeline with deployment!**

---

### Thank You!

Great work completing Session 6! You've learned professional MLOps practices that teams use at scale.

**See you in Session 7 for Model Deployment!**